<a href="https://colab.research.google.com/github/vectara/example-notebooks/blob/main/notebooks/api-examples/9-reranker-instructions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vectara Reranker Instructions with Qwen3

In this notebook we demonstrate how to use **reranker instructions** with Vectara's `qwen3-reranker`. Reranker instructions let you pass domain-specific guidance to the reranker so it can better score relevance for your particular use case.

We'll cover:
- Baseline query using qwen3-reranker **without** instructions
- **Role-based intent steering**: the same query returns different top-5 results depending on whether the user is a researcher or a developer
- **Steering a saturated baseline**: pulling minority-corpus content into the top-5 when vector retrieval strongly favors the opposite side


## About Vectara

[Vectara](https://vectara.com/) is an Agent Platform for trusted enterprise AI — a unified Agentic RAG platform with built-in retrieval, orchestration, and governance. See [Notebook 1](1-corpus-creation.ipynb) for the full overview of features and deployment options.

## Getting Started

This notebook assumes you've completed Notebooks 1 and 2:
- Notebook 1: Created two corpora (ai-research-papers and vectara-docs) with Boomerang embeddings
- Notebook 2: Ingested AI research papers and Vectara documentation

In [1]:
import os
import requests
import json

# Set up authentication
api_key = os.environ['VECTARA_API_KEY']

# Corpus keys from Notebook 1
research_corpus_key = 'tutorial-ai-research-papers'
docs_corpus_key = 'tutorial-vectara-docs'

# Base URL for Vectara API v2
BASE_URL = "https://api.vectara.io/v2"

# Common headers for all requests
headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'x-api-key': api_key
}


def run_query(query_request):
    """Run a Vectara query and print the summary and top results."""
    response = requests.post(f"{BASE_URL}/query", headers=headers, json=query_request)
    if response.status_code != 200:
        print(f"Error: {response.status_code}")
        print(response.text)
        return None

    result = response.json()
    print("\n=== Generated Summary ===")
    print(result['summary'])
    print(f"\n=== Factual Consistency Score: {result.get('factual_consistency_score', 'N/A')} ===")

    print("\n=== Top Search Results ===")
    for i, sr in enumerate(result.get('search_results', [])[:5], 1):
        meta = sr.get('document_metadata', {})
        print(f"\n--- Result {i} (score: {sr.get('score', 'N/A'):.4f}) ---")
        print(f"Document: {sr.get('document_id', 'N/A')}")
        print(f"Title: {meta.get('title', 'N/A')}")
        print(f"Text: {sr['text'][:200]}...")

    return result


print("Setup complete.")

Setup complete.


## Example 1: Baseline Query — Qwen3 Reranker Without Instructions

First, let's run a query using the `qwen3-reranker` **without** any instructions across **both corpora** (research papers and Vectara docs). This establishes a baseline to compare against in the following examples.

Note the API difference from earlier notebooks: instead of `reranker_id`, we use `reranker_name` to select the qwen3 reranker.

In [2]:
QUERY = "How does reranking improve search result quality?"

baseline_request = {
    "query": QUERY,
    "search": {
        "corpora": [
            {
                "corpus_key": research_corpus_key,
                "lexical_interpolation": 0.005
            },
            {
                "corpus_key": docs_corpus_key,
                "lexical_interpolation": 0.005
            }
        ],
        "limit": 100,
        "context_configuration": {
            "sentences_before": 2,
            "sentences_after": 2
        },
        "reranker": {
            "type": "chain",
            "rerankers": [
                {
                    "type": "customer_reranker",
                    "reranker_name": "qwen3-reranker",
                    "limit": 100
                },
                {
                    "type": "mmr",
                    "diversity_bias": 0.05
                }
            ]
        }
    },
    "generation": {
        "generation_preset_name": "vectara-summary-ext-24-05-med-omni",
        "max_used_search_results": 10,
        "response_language": "eng",
        "enable_factual_consistency_score": True
    }
}

print("=" * 60)
print("BASELINE: qwen3-reranker without instructions")
print("=" * 60)
baseline_result = run_query(baseline_request)

BASELINE: qwen3-reranker without instructions



=== Generated Summary ===
Reranking improves search result quality by refining and reordering the results after their initial retrieval. This process enhances the relevance of the results by ensuring that the most pertinent and business-critical results appear at the top. Rerankers can be configured to optimize result quality for different use cases, such as improving precision with neural models, adding diversity, or incorporating custom business logic. They can also provide multilingual support and advanced neural ranking capabilities, making them ideal for applications requiring high relevance and precision [1], [2], [4].

=== Factual Consistency Score: 0.6875 ===

=== Top Search Results ===

--- Result 1 (score: 0.9990) ---
Document: docs-vectara-com-docs-sdk-python-rerankers
Title: Rerankers
Text: Rerankers enhance the relevance of search results by refining and reordering them
after initial retrieval. The Vectara Python SDK enables you to apply various
reranker types in queries 

## Example 2: Role-Based Intent Steering

The same question can mean different things to different users. A researcher writing a survey wants peer-reviewed papers with novel techniques; a developer building a production system wants implementation guides and API references. With **reranker instructions**, the same query against the same two corpora returns a top-5 tailored to each intent — no query rewriting, no corpus filtering, nothing changes upstream of the reranker.

We run the same query three ways and compare the top-5:

1. **Baseline** — qwen3-reranker with no instructions (neutral relevance ordering).
2. **Researcher role** — instructions to prefer peer-reviewed academic papers.
3. **Developer role** — instructions to prefer implementation docs and API references.

The query — *"What techniques reduce hallucinations in LLMs?"* — is deliberately chosen to sit in the overlap between the two corpora: the research corpus has academic papers on hallucination mitigation, and the Vectara docs cover grounding and factual-consistency features. That mixed baseline is exactly the situation where instructions have room to steer.


In [3]:
# Query with a mixed baseline: both corpora have relevant content, so the
# reranker has headroom to steer in either direction based on the instructions.
ROLE_QUERY = "What techniques reduce hallucinations in LLMs?"


def role_request(instructions=None):
    """Build a query request, optionally attaching reranker instructions."""
    reranker = {
        "type": "customer_reranker",
        "reranker_name": "qwen3-reranker",
        "limit": 100,
        "cutoff": 0.2,
    }
    if instructions:
        reranker["instructions"] = instructions
    return {
        "query": ROLE_QUERY,
        "search": {
            "corpora": [
                {"corpus_key": research_corpus_key, "lexical_interpolation": 0.005},
                {"corpus_key": docs_corpus_key, "lexical_interpolation": 0.005},
            ],
            "limit": 100,
            "context_configuration": {"sentences_before": 2, "sentences_after": 2},
            "reranker": {
                "type": "chain",
                "rerankers": [reranker, {"type": "mmr", "diversity_bias": 0.05}],
            },
        },
        "generation": {
            "generation_preset_name": "vectara-summary-ext-24-05-med-omni",
            "max_used_search_results": 10,
            "response_language": "eng",
            "enable_factual_consistency_score": True,
        },
    }


RESEARCHER_INSTRUCTIONS = (
    "The user is an academic researcher writing a survey paper. Prioritize "
    "peer-reviewed academic research papers with novel techniques, experiments, "
    "and benchmark results. Deprioritize product documentation and tutorials."
)
DEVELOPER_INSTRUCTIONS = (
    "The user is a software developer building a production search application. "
    "Prioritize practical implementation guides, API reference, configuration "
    "options, and code examples. Deprioritize academic theory and research papers."
)

print("=" * 60)
print("BASELINE — no instructions")
print("=" * 60)
baseline_role_result = run_query(role_request())

print("\n" + "=" * 60)
print("RESEARCHER — prioritize academic papers")
print("=" * 60)
researcher_result = run_query(role_request(RESEARCHER_INSTRUCTIONS))

print("\n" + "=" * 60)
print("DEVELOPER — prioritize implementation docs")
print("=" * 60)
developer_result = run_query(role_request(DEVELOPER_INSTRUCTIONS))


BASELINE — no instructions



=== Generated Summary ===
Techniques to reduce hallucinations in large language models (LLMs) include self-reflection mechanisms, as explored by Ziwei Ji et al., which involve the model evaluating its own outputs to identify and correct errors [1]. Additionally, retrieval-augmented generation (RAG) techniques are used to ground the model's outputs in trusted source content, thereby improving factual accuracy [6]. The Hallucination Correctors API is another tool that detects and corrects factual inaccuracies by comparing generated summaries against source documents [6]. Furthermore, breaking down summaries into sentences or claims before detecting hallucinations, as done by RAGAS and TruLens, has shown to improve accuracy [8].

=== Factual Consistency Score: 0.93359375 ===

=== Top Search Results ===

--- Result 1 (score: 0.9995) ---
Document: hallucination-detection-naacl.pdf
Title: Hallucination Detection in RAG Systems
Text: **Mitigation Approaches (2023)**: Ziwei Ji et al. also exp


=== Generated Summary ===
Techniques to reduce hallucinations in large language models (LLMs) include the use of retrieval-augmented generation, which integrates external knowledge sources to improve factual consistency [1]. Another approach is the implementation of self-reflection mechanisms, allowing models to evaluate and correct their outputs [3]. Additionally, developing benchmarks and empirical analyses, such as HalluDIAL and ANAH, helps in understanding and mitigating hallucinations by providing structured evaluation frameworks [8]. These methods collectively aim to enhance the reliability and factual accuracy of LLMs.

=== Factual Consistency Score: 0.19433594 ===

=== Top Search Results ===

--- Result 1 (score: 0.9740) ---
Document: hallucination-detection-naacl.pdf
Title: Hallucination Detection in RAG Systems
Text: **ChatGPT as an Evaluator (2023)**: Zheheng Luo et al. propose using ChatGPT for evaluating factual inconsistencies specifically in text summarization tasks. **


=== Generated Summary ===
Techniques to reduce hallucinations in large language models (LLMs) include the use of hallucination correctors, which automatically detect and correct factual inaccuracies by comparing generated summaries against source documents, ensuring outputs are grounded in trusted content [1]. Additionally, self-reflection mechanisms have been explored as a method to mitigate hallucinations [2]. Retrieval Augmented Generation (RAG) pipelines are also employed to improve factual accuracy by grounding responses in reliable source material [3].

=== Factual Consistency Score: 0.94921875 ===

=== Top Search Results ===

--- Result 1 (score: 0.8725) ---
Document: docs-vectara-com-docs-rest-api-correct-hallucinations
Title: Corrects hallucinations in generated text based on source documents
Text: Code example: POST/v2/hallucination_correctors/correct_hallucinations The Hallucination Correctors API enables users to automatically detect and correct factual inaccuracies, commo

In [4]:
def classify_result(doc_id):
    """Classify a result as 'paper' or 'docs' based on document ID."""
    return 'paper' if doc_id.endswith('.pdf') else 'docs'


print(f"=== Role-Based Intent Steering — top-5 breakdown ===")
print(f"Query: {ROLE_QUERY!r}\n")

for label, result in [("BASELINE         ", baseline_role_result),
                      ("RESEARCHER role  ", researcher_result),
                      ("DEVELOPER role   ", developer_result)]:
    top5 = result.get('search_results', [])[:5]
    papers = sum(1 for sr in top5 if classify_result(sr['document_id']) == 'paper')
    docs = len(top5) - papers
    print(f"{label}  papers/docs = {papers}/{docs}")
    for i, sr in enumerate(top5, 1):
        kind = classify_result(sr['document_id'])
        print(f"  {i}. [{kind:5s}] {sr['document_id']} (score: {sr.get('score', 0):.4f})")
    print()

print("-> Same query, same corpora, same retrieval layer. The instruction")
print("   alone shifts which content rises to the top of the results.")


=== Role-Based Intent Steering — top-5 breakdown ===
Query: 'What techniques reduce hallucinations in LLMs?'

BASELINE           papers/docs = 5/0
  1. [paper] hallucination-detection-naacl.pdf (score: 0.9995)
  2. [paper] hallucination-detection-naacl.pdf (score: 0.9092)
  3. [paper] hallucination-detection-naacl.pdf (score: 0.9085)
  4. [paper] hallucination-detection-naacl.pdf (score: 0.9067)
  5. [paper] hallucination-detection-naacl.pdf (score: 0.9060)

RESEARCHER role    papers/docs = 5/0
  1. [paper] hallucination-detection-naacl.pdf (score: 0.9740)
  2. [paper] hallucination-detection-naacl.pdf (score: 0.8826)
  3. [paper] hallucination-detection-naacl.pdf (score: 0.8576)
  4. [paper] hallucination-detection-naacl.pdf (score: 0.8507)
  5. [paper] hallucination-detection-naacl.pdf (score: 0.8450)

DEVELOPER role     papers/docs = 3/2
  1. [docs ] docs-vectara-com-docs-rest-api-correct-hallucinations (score: 0.8725)
  2. [paper] hallucination-detection-naacl.pdf (score: 0.7399)
 

## Example 3: Steering a Saturated Baseline

The previous example used a query where both corpora surfaced in the baseline top-5 — there was a natural mix to rearrange. A more common production scenario is that vector retrieval is already **saturated** in one direction: all your top results come from one subset of your corpus, and the content your user actually needs is buried below the top-N.

Here we show that reranker instructions can **pull a minority subset of the corpus into the top-5 even when vector retrieval strongly favors the opposite side**. The query `"chunking strategies for long documents"` is an all-docs query by default — Vectara's ingestion docs describe chunking extensively, so they dominate. Academic chunking research exists in the research corpus but never makes it into the top-5 without help.

We reuse the researcher instruction from Example 2 to boost academic papers into the saturated top-5.


In [5]:
# Query where the baseline retrieval is saturated toward Vectara docs.
SATURATED_QUERY = "chunking strategies for long documents"


def saturated_request(instructions=None):
    """Build a query request for the saturated-baseline demo."""
    reranker = {
        "type": "customer_reranker",
        "reranker_name": "qwen3-reranker",
        "limit": 100,
        "cutoff": 0.2,
    }
    if instructions:
        reranker["instructions"] = instructions
    return {
        "query": SATURATED_QUERY,
        "search": {
            "corpora": [
                {"corpus_key": research_corpus_key, "lexical_interpolation": 0.005},
                {"corpus_key": docs_corpus_key, "lexical_interpolation": 0.005},
            ],
            "limit": 100,
            "context_configuration": {"sentences_before": 2, "sentences_after": 2},
            "reranker": {
                "type": "chain",
                "rerankers": [reranker, {"type": "mmr", "diversity_bias": 0.05}],
            },
        },
        "generation": {
            "generation_preset_name": "vectara-summary-ext-24-05-med-omni",
            "max_used_search_results": 10,
            "response_language": "eng",
            "enable_factual_consistency_score": True,
        },
    }


print("=" * 60)
print("BASELINE — no instructions (vector retrieval saturated with docs)")
print("=" * 60)
saturated_baseline_result = run_query(saturated_request())


BASELINE — no instructions (vector retrieval saturated with docs)



=== Generated Summary ===
Chunking strategies for long documents involve dividing the text into manageable parts for better processing and retrieval. Common strategies include:

1. **Sentence Chunking**: This strategy creates one chunk per sentence, which is the default method. It provides optimal retrieval accuracy for most datasets [1], [6].

2. **Max-Chars Chunking**: This approach accumulates sentences into a chunk until a specified character limit is reached. If a single sentence exceeds this limit, it is split across chunks. This method balances retrieval speed with contextual coherence [1], [6].

3. **Recursive Chunking**: This involves splitting the document by paragraph, then sentence, and finally word, until a size target is met [3].

4. **Semantic Chunking**: This strategy splits the document where sentence-level similarity drops, maintaining semantic integrity [3].

5. **Sliding-Window Chunking**: This method uses fixed-size chunks with token overlap between adjacent chunk

In [6]:
# Same query, same corpora, same reranker. Only the instruction changes.
print("=" * 60)
print("WITH RESEARCHER INSTRUCTION — academic chunking papers boosted")
print("=" * 60)
saturated_researcher_result = run_query(saturated_request(RESEARCHER_INSTRUCTIONS))


WITH RESEARCHER INSTRUCTION — academic chunking papers boosted



=== Generated Summary ===
Chunking strategies for long documents include several approaches to effectively manage and process text. Some common strategies are:

1. **Recursive Chunking**: This involves splitting the document by paragraph, then sentence, and finally word, until a desired size is achieved [3].

2. **Semantic Chunking**: This method splits the document where there is a drop in sentence-level similarity, ensuring that each chunk maintains semantic coherence [3].

3. **Sliding-Window Chunking**: This approach creates fixed-size chunks with overlapping tokens between adjacent chunks, which helps in maintaining context across chunks [3].

4. **Hierarchical Chunking**: This strategy indexes the same content at different granularities, providing both context and precision [3].

5. **Max-Chars Chunking**: This accumulates sentences up to a character limit, balancing context richness against the risk of diluting the meaning by spanning multiple topics [6].

These strategies help

In [7]:
print(f"=== Saturated-Baseline Steering — top-5 breakdown ===")
print(f"Query: {SATURATED_QUERY!r}\n")

for label, result in [("BASELINE            ", saturated_baseline_result),
                      ("WITH RESEARCHER     ", saturated_researcher_result)]:
    top5 = result.get('search_results', [])[:5]
    papers = sum(1 for sr in top5 if classify_result(sr['document_id']) == 'paper')
    docs = len(top5) - papers
    print(f"{label}  papers/docs = {papers}/{docs}")
    for i, sr in enumerate(top5, 1):
        kind = classify_result(sr['document_id'])
        print(f"  {i}. [{kind:5s}] {sr['document_id']} (score: {sr.get('score', 0):.4f})")
    print()

# Measure the set churn: how many unique docs entered or left the top-5.
baseline_ids = {sr['document_id'] for sr in saturated_baseline_result.get('search_results', [])[:5]}
researcher_ids = {sr['document_id'] for sr in saturated_researcher_result.get('search_results', [])[:5]}
swapped_in = researcher_ids - baseline_ids
swapped_out = baseline_ids - researcher_ids
print(f"Top-5 churn: {len(swapped_in)} document(s) entered, {len(swapped_out)} left.")
print()
print("-> Without instructions, vector retrieval never surfaces research papers")
print("   for this query — Vectara's chunking docs dominate the top-5. A single")
print("   role instruction is enough to pull academic chunking research into")
print("   the top results even though the query itself is unchanged.")


=== Saturated-Baseline Steering — top-5 breakdown ===
Query: 'chunking strategies for long documents'

BASELINE              papers/docs = 0/5
  1. [docs ] docs-vectara-com-docs-rest-api-create-corpus-document (score: 0.9997)
  2. [docs ] docs-vectara-com-docs-build-chunking-strategies (score: 0.9079)
  3. [docs ] docs-vectara-com-docs-build-chunking-strategies (score: 0.9073)
  4. [docs ] docs-vectara-com-docs-build-chunking-strategies (score: 0.9069)
  5. [docs ] docs-vectara-com-docs-build-chunking-strategies (score: 0.9050)

WITH RESEARCHER       papers/docs = 2/3
  1. [paper] rag-retrieval-augmented-generation.pdf (score: 0.9520)
  2. [paper] rag-retrieval-augmented-generation.pdf (score: 0.8056)
  3. [docs ] docs-vectara-com-docs-build-chunking-strategies (score: 0.7675)
  4. [docs ] docs-vectara-com-docs-build-chunking-strategies (score: 0.7225)
  5. [docs ] docs-vectara-com-docs-platform-architecture-platform-stack (score: 0.6941)

Top-5 churn: 2 document(s) entered, 1 left.

-